In [ ]:
# SABX__dryrun.ipynb
# Цель: черновик безопасного обработчика задач PCReboot.
#
# Здесь мы НЕ делаем WinRM.
# Здесь мы принимаем RebootTask и возвращаем RebootResult.
# Пока только dry-run и проверка allowed_hosts.

Жизненный цикл (ЖЦ программы) при статус ПК = выкл:
1. Задача создана          → PENDING
2. Проверяем allowed_hosts → прошёл
3. Проверяем доступность   → ping не идёт, порт 5985 закрыт
4. Останавливаемся         → PRECHECK_FAILED

PENDING
  ↓
allowed_hosts? ──нет──→ NOT_ALLOWED
  ↓ да
dry_run? ──да──→ SKIPPED
  ↓ нет
pre-check (ping / порт) ──недоступен──→ PRECHECK_FAILED
  ↓ доступен
WinRM auth ──ошибка──→ AUTH_ERROR
  ↓ ок
отправка shutdown ──ошибка──→ COMMAND_ERROR
  ↓ отправлена
COMMAND_SENT
  ↓
наблюдение 60 сек ──пропал──→ REBOOT_CONFIRMED
  ↓ не пропал
TIMEOUT

In [ ]:
ALLOWED_HOSTS = (
    "WS-K534D",
    "WS-K534F",
)

1. Создать RebootResult со статусом PENDING.
2. Проверить, что host есть в ALLOWED_HOSTS.
3. Если нет — вернуть NOT_ALLOWED.
4. Если dry_run=True — вернуть SKIPPED с сообщением, что команда была бы отправлена.
5. Если dry_run=False — пока вернуть UNKNOWN_ERROR или COMMAND_ERROR с пометкой, что реальный backend ещё не подключён.

In [ ]:
from core.models import RebootTask, RebootResult, RebootStatus
from datetime import datetime

# TODO: создать базовый RebootResult со статусом PENDING:
dry_res = RebootResult(host="WS-73939", run_id="run001", status=RebootStatus.PENDING, message="Ожидаем выполнения...", started_at= datetime.now())
def process_task(task: RebootTask, allowed_hosts: set[str]) -> RebootResult:
    """
    Обрабатывает одну задачу перезагрузки.

    Пока только:
    - проверка allowed_hosts;
    - dry-run;
    - без реального WinRM.
    """
    
    # MADE: если task.host нет в allowed_hosts, вернуть NOT_ALLOWED
    if task.host not in allowed_hosts:
        task.dry_run = RebootStatus.PENDING
    
    # MADE: если task.dry_run, вернуть SKIPPED
    # MADE: message должен показывать команду, которая была бы выполнена:
    # shutdown /r /t {task.reboot_delay_sec} /f
    if task.dry_run:
        mess = f"SKIPPED shutdown /r /t {task.reboot_delay_sec} /f"
        print(mess)
        task.dry_run = RebootStatus.SKIPPED
        

    # MADE: если не dry-run, пока вернуть COMMAND_ERROR
    # message: "Real backend not implemented yet"
    if not task.dry_run:
        task.dry_run = RebootStatus.COMMAND_ERROR



